In [ ]:
# اجرا در Google Colab
!pip -q install gdown beir

import os
import tarfile
import gdown
import pandas as pd
import numpy as np

DATA_DIR = "/content/hamshahri2"
ARCHIVE_PATH = "/content/dataframe_hamshahri2.tar"
DRIVE_FILE_ID = "1N5V40HQsnCHHytdmG98YfXvN_3DCjV67"

if not os.path.exists(os.path.join(DATA_DIR, "docs.csv.gz")):
    if not os.path.exists(ARCHIVE_PATH):
        gdown.download(id=DRIVE_FILE_ID, output=ARCHIVE_PATH, quiet=False)
    with tarfile.open(ARCHIVE_PATH) as tar:
        tar.extractall(path="/content")

df_docs = pd.read_csv(os.path.join(DATA_DIR, "docs.csv.gz"), compression="gzip")
df_judgments = pd.read_csv(os.path.join(DATA_DIR, "judgments_dataframe.csv.gz"), compression="gzip")
df_queries_fa = pd.read_csv(os.path.join(DATA_DIR, "queries_fa.csv.gz"), compression="gzip")

print("Documents:", len(df_docs))
print("Queries:", len(df_queries_fa))
print("Judgments:", len(df_judgments))
df_docs.head(2)

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import normalize

# نگه‌داشتن همان ترتیب اسناد و شناسه‌های dataframe مطابق نوت‌بوک مرجع
docs = df_docs["TITLE"].fillna("").astype(str).tolist()
queries = df_queries_fa["TITLE"].fillna("").astype(str).tolist()

qrels = {}
for query_id, group in df_judgments[df_judgments["relevancy"] == 1].groupby("query_id"):
    qrels[str(query_id)] = {
        str(doc_id): int(rel)
        for doc_id, rel in zip(group["doc_id"], group["relevancy"])
    }

vectorizer = TfidfVectorizer()
tfidf_docs = vectorizer.fit_transform(docs)
tfidf_queries = vectorizer.transform(queries)

print("TF-IDF matrix:", tfidf_docs.shape)
print("Relevant-query groups:", len(qrels))

In [ ]:
# همان محاسبه مرجع؛ همه queryها با همه اسناد
baseline_scores = cosine_similarity(tfidf_queries, tfidf_docs)

def build_ranked_results(score_matrix, k=1000):
    ranked = {}
    for row_idx, scores in enumerate(score_matrix):
        top_indices = np.argsort(scores)[-k:][::-1]
        ranked[str(row_idx + 1)] = {
            str(df_docs.iloc[doc_idx]["DOCID"]): float(scores[doc_idx])
            for doc_idx in top_indices
        }
    return ranked

baseline_results = build_ranked_results(baseline_scores, k=1000)
print("Baseline query count:", len(baseline_results))

In [ ]:
# normalize روی ماتریس sparse انجام می‌شود و از تبدیل به آرایه dense جلوگیری می‌کند.
tfidf_docs_l2 = normalize(tfidf_docs, norm="l2", axis=1)
tfidf_queries_l2 = normalize(tfidf_queries, norm="l2", axis=1)

# شباهت فقط با ضرب نقطه‌ای؛ هیچ تابع CosineSimilarity در این مرحله استفاده نمی‌شود.
manual_scores_sparse = tfidf_queries_l2 @ tfidf_docs_l2.T
manual_scores = manual_scores_sparse.toarray()

manual_results = build_ranked_results(manual_scores, k=1000)

max_abs_diff = np.max(np.abs(baseline_scores - manual_scores))
print("Maximum absolute score difference:", max_abs_diff)
print("All scores equal within 1e-10:",
      np.allclose(baseline_scores, manual_scores, atol=1e-10))

In [ ]:
# بررسی اینکه ترتیب 1000 نتیجه هر پرس‌وجو در دو روش یکسان است
same_rankings = all(
    list(baseline_results[qid].keys()) == list(manual_results[qid].keys())
    for qid in baseline_results
)
print("All top-1000 rankings identical:", same_rankings)

from beir.retrieval.evaluation import EvaluateRetrieval

top_k_values = [1, 3, 5, 10, 100, 1000]
baseline_metrics = EvaluateRetrieval.evaluate(qrels, baseline_results, top_k_values)
manual_metrics = EvaluateRetrieval.evaluate(qrels, manual_results, top_k_values)

def metrics_to_df(metrics, method_name):
    rows = []
    for metric_group in metrics:
        for metric_name, score in metric_group.items():
            metric, k = metric_name.split("@")
            rows.append({"Method": method_name, "k": int(k),
                         "Metric": metric, "Score": float(score)})
    return pd.DataFrame(rows)

metrics_comparison = pd.concat([
    metrics_to_df(baseline_metrics, "Baseline cosine"),
    metrics_to_df(manual_metrics, "L2 normalization + dot product")
], ignore_index=True)

metrics_pivot = metrics_comparison.pivot_table(
    index=["k", "Metric"], columns="Method", values="Score"
).reset_index()
metrics_pivot["Absolute difference"] = (
    metrics_pivot["Baseline cosine"] -
    metrics_pivot["L2 normalization + dot product"]
).abs()
metrics_pivot